In [1]:
# %pip install xgboost

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv('../After_EDA_data/Light_text.csv')

In [4]:
df

,uid,profile,anime_uid,score,scores,text
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...
...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...


In [5]:
def classification(score):
    if score > 6:
        return 'Good'
    elif score > 4:
        return 'Neutral'
    else:
        return 'Bad'

In [6]:
df['target'] = df['score'].apply(classification)

In [7]:
df

,uid,profile,anime_uid,score,scores,text,target
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...,Good
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...,Good
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...,Good
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...,Good
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...,Good
...,...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...,Good
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...,Good
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...,Bad
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...,Good


In [8]:
df['target'].value_counts()

target
Good       142343
Neutral     27651
Bad         22118
Name: count, dtype: int64

In [9]:
df_majority = df[df['target'] == 'Good']
df_minority = df[df['target'] == 'Bad']
df_neutral = df[df['target'] == 'Neutral']

In [10]:
df_reduced_majority = df_majority.sample(n=len(df_neutral), random_state=42)

In [11]:
df_reduced_majority.value_counts()

uid     profile           anime_uid  score  scores                                                                                                   text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [12]:
df_balanced = pd.concat([df_reduced_majority, df_minority, df_neutral]).sample(frac=1, random_state=42)

In [13]:
df_balanced['target'].value_counts()

target
Good       27651
Neutral    27651
Bad        22118
Name: count, dtype: int64

In [14]:
df_balanced = df_balanced.reset_index(drop=True)

In [15]:
df_balanced

,uid,profile,anime_uid,score,scores,text,target
0,13952,KillerMan91,73,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",full metal panic! tsr is 13 episodes long sequ...,Good
1,305201,Grizzziff,21877,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",what a clever little nuance factory cash grab ...,Good
2,243817,Fircoal,33089,8,"{'Overall': '8', 'Story': '9', 'Animation': '2...",i have a confession to make when i watched the...,Good
3,284471,Botmj,36214,4,"{'Overall': '4', 'Story': '1', 'Animation': '1...",the only good thing i have to say about this i...,Bad
4,291301,BabyGirl06301,31765,8,"{'Overall': '8', 'Story': '9', 'Animation': '1...",i thought this movie was a wonderful addition ...,Good
...,...,...,...,...,...,...,...
77415,141887,Flueckli,10098,7,"{'Overall': '7', 'Story': '6', 'Animation': '4...",don t expect too much out of these shortstorie...,Good
77416,226574,FAKEANIMEGIRL,2724,6,"{'Overall': '6', 'Story': '0', 'Animation': '0...",gainax s two short films made for the science ...,Neutral
77417,241425,Rizkiawan-kun,21995,6,"{'Overall': '6', 'Story': '7', 'Animation': '6...",in short this anime is not more than one shouj...,Neutral
77418,18387,boundtotomorrow,853,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",to put it simply i wish more if not all animes...,Good


In [16]:
x = df_balanced['text']
y = df_balanced['target']

In [17]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(le.classes_)

['Bad' 'Good' 'Neutral']


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

In [19]:
df_balanced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77420 entries, 0 to 77419
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   uid        77420 non-null  int64 
 1   profile    77420 non-null  object
 2   anime_uid  77420 non-null  int64 
 3   score      77420 non-null  int64 
 4   scores     77420 non-null  object
 5   text       77420 non-null  object
 6   target     77420 non-null  object
dtypes: int64(3), object(4)
memory usage: 4.1+ MB


In [20]:
tfidf = TfidfVectorizer(
    max_features=1000,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [21]:
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.5,
    subsample=0.8,
    colsample_bytree=0.5,
    eval_metric='logloss'
)

model.fit(X_train_tfidf, y_train)


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.5
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [22]:
y_pred = model.predict(X_test_tfidf)
y_pred_labels = le.inverse_transform(y_pred)
print(y_pred_labels)

['Good' 'Neutral' 'Bad' ... 'Neutral' 'Good' 'Bad']


In [23]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

         Bad       0.73      0.67      0.70      6636
        Good       0.78      0.80      0.79      8295
     Neutral       0.64      0.67      0.66      8295

    accuracy                           0.72     23226
   macro avg       0.72      0.71      0.72     23226
weighted avg       0.72      0.72      0.72     23226



In [24]:
print(confusion_matrix(y_test, y_pred))

[[4474  446 1716]
 [ 314 6614 1367]
 [1335 1403 5557]]


In [25]:
y_prob = model.predict_proba(X_test_tfidf)

In [26]:
roc_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class='ovr',
    average='macro'
)

print("ROC AUC:", roc_auc)

ROC AUC: 0.8773314187193982
